In [1]:
import pandas as pd

# load scraped data
df = pd.read_csv("odi_player_stats.csv")

print("Shape before cleaning:", df.shape)

# remove extra spaces from player names
df["Player"] = df["Player"].str.strip()

# remove duplicate players if any (can happen due to pagination overlap)
df = df.drop_duplicates(subset="Player")

# replace missing value markers used by the website
df = df.replace(["-", "DNB", "TDNB", "absent"], pd.NA)

# convert every column except Player to numeric
for col in df.columns:
    if col != "Player":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# drop players with no stats at all (neither batting nor bowling)
stat_cols = [c for c in df.columns if c != "Player"]
df = df.dropna(subset=stat_cols, how="all")

# drop players with very few matches - unreliable stats, skews visualizations
if "bat_Mat" in df.columns:
    df = df[(df["bat_Mat"].isna()) | (df["bat_Mat"] >= 5)]

# fill missing numeric stats with 0 where it makes sense (e.g. a pure batsman
# will have NaN for all bowling columns - that's fine to leave as NaN,
# but 100s/50s/wickets being NaN for players who did play should become 0)
count_cols = [c for c in df.columns if any(k in c for k in ["100", "50", "0", "Wkts"])]
for col in count_cols:
    df[col] = df[col].fillna(0)

df = df.reset_index(drop=True)

print("Shape after cleaning:", df.shape)
print(df.head())

df.to_csv("odi_clean_dataset.csv", index=False)
print("\nSaved: odi_clean_dataset.csv")

Shape before cleaning: (440, 27)
Shape after cleaning: (440, 27)
        Player  bat_Span  bat_Mat  bat_Inns  bat_NO  bat_Runs  bat_HS  \
0  A Balbirnie       NaN    117.0     110.0     8.0    3264.0     NaN   
1   A Flintoff       NaN    141.0     122.0    16.0    3394.0   123.0   
2     A Flower       NaN    213.0     208.0    16.0    6786.0   145.0   
3     A Jadeja       NaN    196.0     179.0    36.0    5359.0   119.0   
4     A Kumble       NaN      NaN       NaN     NaN       NaN     NaN   

   bat_Ave  bat_BF  bat_SR  ...  bowl_Balls  bowl_Runs  bowl_Wkts  bowl_BBI  \
0    32.00  4396.0   74.24  ...         NaN        NaN        0.0       NaN   
1    32.01  3821.0   88.82  ...      5624.0     4121.0      169.0       NaN   
2    35.34  9097.0   74.59  ...         NaN        NaN        0.0       NaN   
3    37.47  7678.0   69.79  ...         NaN        NaN        0.0       NaN   
4      NaN     NaN     NaN  ...     14496.0    10412.0      337.0       NaN   

   bowl_Ave  bowl_Eco